In [ ]:
import speedtest

def check_internet_speed():
    """
    Checks and prints the internet download and upload speed.
    """
    st = speedtest.Speedtest()
    print("Getting best server...")
    st.get_best_server()
    print(f"Testing download speed...")
    download_speed = st.download()
    print(f"Testing upload speed...")
    upload_speed = st.upload()

    # Convert speeds from bits per second to megabits per second (Mbps)
    download_mbps = download_speed / 1_000_000
    upload_mbps = upload_speed / 1_000_000

    print(f"\nDownload Speed: {download_mbps:.2f} Mbps")
    print(f"Upload Speed: {upload_mbps:.2f} Mbps")
    print(f"Ping: {st.results.ping:.2f} ms")

check_internet_speed()

In [ ]:
import earthaccess
import os
import re

In [ ]:
earthaccess.login()

In [ ]:
os.chdir('/g/data/gv90/ae8456/SWOT_SEAICE/DATA')

In [ ]:
help(earthaccess.search_data)

In [ ]:
#Thwaites Glacier 
#From sow_l3_data_all

half_orbits = [5,
 7,
 33,
 61,
 89,
 117,
 145,
 173,
 201,
 229,
 255,
 257,
 283,
 285,
 311,
 313,
 339,
 367,
 395,
 423,
 451,
 479,
 507,
 535,
 561,
 563, 584, 582, 556,554,528,500,472,444,424,416,388,360,332,206,304,278,276,250,222,194,166,138,110,82,78,54,28,26]

In [ ]:
# TEST

strep = f"{half_orbits[0]:03}"
gran_name =  f"*Unsmoothed_*_{strep}_*"
print(gran_name)

In [ ]:
# TEST

# # concept_id="C2799465497-POCLOUD", 

# Thwaites Glacier
results = earthaccess.search_data(
    #short_name="SWOT_L2_LR_SSH_2.0",
    short_name="SWOT_L2_LR_SSH_D",
    granule_name = gran_name,
    orbit_number = [],
    temporal=("2025-03-01","2026-01-01"),
    bounding_box=(-113,-75.5,-100, -72),
    )

print(len(results))

In [ ]:
# files = earthaccess.download(
#     results,
#     local_path="SWOT_L2_Unsmoothed_Full",
#     provider="POCLOUD",
#     threads=4
# )

In [ ]:
# Convert the results list to a single string
results_str = " ".join(map(str, results))
# Regex pattern to match URLs inside Data: [...]
pattern = r"Data:\s*\[(.*?)\]"
# Find all matches
matches = re.findall(pattern, results_str)
# Each match may contain multiple URLs separated by commas
all_urls = []
for match in matches:
    # Remove quotes and whitespace, split by comma
    urls = [url.strip().strip("'\"") for url in match.split(",")]
    all_urls.extend(urls)
# print(all_urls)
filenames = [url.split("/")[-1] for url in all_urls]
print(filenames)

In [ ]:
# out dir
data_dir = "SWOT_L2_Unsmoothed_Full"
os.makedirs(data_dir, exist_ok=True)  
# Loop over all half orbits

for half_orbit in half_orbits[0:]:
    strep = f"{half_orbit:03}"
    gran_name = f"*Unsmoothed_*_{strep}_*"
    print(f"Searching for granule: {gran_name}")
    # Thwaites Glacier
    results = earthaccess.search_data(
        # short_name="SWOT_L2_LR_SSH_2.0", # version C
        short_name="SWOT_L2_LR_SSH_D",
        granule_name = gran_name,
        orbit_number = [],
        temporal=("2025-03-01","2026-01-01"),
        bounding_box=(-113,-75.5,-100, -72),
        )
    """
    # Mawson Coast
    results = earthaccess.search_data(
        # short_name="SWOT_L2_LR_SSH_2.0", # version C
        short_name="SWOT_L2_LR_SSH_D", # version D
        granule_name=gran_name,
        orbit_number=[],
        temporal=("2023-07-01", "2025-12-31"),
        bounding_box=(50, -70, 75, -60),
    )
    """

    """
    # Kapp Norvegia
    results = earthaccess.search_data(
        # short_name="SWOT_L2_LR_SSH_2.0",
        short_name="SWOT_L2_LR_SSH_D",
        granule_name = gran_name,
        orbit_number = [],
        temporal=("2025-11-15","2026-01-10"),
        bounding_box=(-18,-73,-3, -68),
        )
    """

    """
    # Svalbard
    results = earthaccess.search_data(
        short_name="SWOT_L2_LR_SSH_2.0",
        # short_name="SWOT_L2_LR_SSH_D",
        granule_name = gran_name,
        orbit_number = [],
        temporal=("2023-07-01","2025-12-31"),
        bounding_box=(8,76,28, 82),
        )
    """
    
    print(f"Found {len(results)} results for half_orbit {half_orbit}")

    # Convert the results list to a single string
    results_str = " ".join(map(str, results))
    # Regex pattern to match URLs inside Data: [...]
    pattern = r"Data:\s*\[(.*?)\]"
    # Find all matches
    matches = re.findall(pattern, results_str)
    # Each match may contain multiple URLs separated by commas
    all_urls = []
    for match in matches:
        # Remove quotes and whitespace, split by comma
        urls = [url.strip().strip("'\"") for url in match.split(",")]
        all_urls.extend(urls)
    # print(all_urls)
    filenames = [url.split("/")[-1] for url in all_urls]

    # list of existing files in directory
    dir_files = os.listdir(data_dir)
    
    mask = []
    
    for fname in filenames:
        # extract the identifying part (before the timestamp)
        key = "_".join(fname.split("_")[:7])
        print(key)
        # example result: SWOT_L2_LR_SSH_Unsmoothed_012_355
    
        # check if any file in directory contains this key
        exists = any(key in f for f in dir_files)
    
        # 0 = file exists, 1 = no file
        mask.append(0 if exists else 1)

    # OPTIONAL: don't download a file with any version of the same file already in data_dir
    filtered_results = [r for r, m in zip(results, mask) if m == 1]

    # if results:
    if filtered_results:
        files = earthaccess.download(
            filtered_results,
            local_path=data_dir,
            provider="POCLOUD",
            threads=4
        )
        print(f"Downloaded {len(files)} files for half_orbit {half_orbit}")
    else:
        print(f"No files found for half_orbit {half_orbit}")

In [ ]:
def delete_unmatched_pass_files(directory, filtered, dry_run=True):
    """
    Print (dry run) or delete files whose pass number is not in the filtered swath_geometries.
    
    Parameters
    ----------
    directory : str
        Path to directory containing SWOT .nc files
    filtered : list of half orbits
    dry_run : bool
        If True, only print what would be deleted. If False, delete the files.
    """

    # Convert all pass numbers in filtered to integers
    valid_passes = filtered

    for filename in os.listdir(directory):
        if not filename.endswith(".nc"):
            continue

        parts = filename.split('_')
        if len(parts) < 7:
            continue  # skip malformed filenames

        # Extract pass number from filename and convert to int
        try:
            pass_num = int(parts[6])  # 6th index (7th field) is the pass number
        except ValueError:
            continue  # skip if not an integer

        # Delete if pass number is NOT in filtered
        if pass_num not in valid_passes:
            file_path = os.path.join(directory, filename)
            if dry_run:
                print(f"[DRY RUN] Would delete: {file_path}")
            else:
                print(f"Deleting: {file_path}")
                os.remove(file_path)


In [ ]:
# In case existing files in dir not matching filetred pass numbers need to be deleted
L2_outdir = "/g/data/gv90/ae8456/SWOT_SEAICE/DATA/SWOT_L2_Unsmoothed_Full/"
delete_unmatched_pass_files(L2_outdir, half_orbits, dry_run=True) # turn dry_run to False
